## COE Price Prediction & Quota Elasticity Project

### Problem Statement

Singapore’s COE market is highly volatile. Prices for Categories A & B remain elevated, partly due to fluctuations in COE quotas, which determine how many new vehicles can be registered.

Every quarter, LTA announces new quotas → bidders adjust behaviour → COE prices change.

### Objective
Build a predctive model to forecast COE prices (Cat A & B), and quantify quote elasticity i.e., How much do COE prices change when quote increases/decreases by X?


### Data Preparation

In [10]:
import pandas as pd

print("COE Quota...")
coe_quota_df = pd.read_csv("MotorVehicleQuotaQuotaPremiumAndPrevailingQuotaPremiumMonthly.csv")

display(coe_quota_df.head(2))
print(coe_quota_df.columns.to_list()[:10])
print(coe_quota_df.shape)

print("COE Price...")
coe_price_df = pd.read_csv("COEBiddingResultsPrices.csv")
display(coe_price_df.head(2))
print(coe_price_df.shape)

COE Quota...


,DataSeries,2025Oct,2025Sep,2025Aug,2025Jul,2025Jun,2025May,2025Apr,2025Mar,2025Feb,...,2002Nov,2002Oct,2002Sep,2002Aug,2002Jul,2002Jun,2002May,2002Apr,2002Mar,2002Feb
0,"Total Quota, 1st Bidding",3164,3144,3141,3070,3086,3094,2882,2892,2869,...,4617,3932,3936,3935,3952,4060,3990,3695,3955,3939
1,"Total Successful Bids, 1st Bidding",3119,3097,3114,3046,3054,3048,2828,2865,2833,...,4600,3804,3914,3909,3927,4018,3840,3615,3943,3910


['DataSeries', '2025Oct', '2025Sep', '2025Aug', '2025Jul', '2025Jun', '2025May', '2025Apr', '2025Mar', '2025Feb']
(50, 286)
COE Price...


,month,bidding_no,vehicle_class,quota,bids_success,bids_received,premium
0,2010-01,1,Category A,1152,1145,1342,18502
1,2010-01,1,Category B,687,679,883,19190


(1880, 7)


In [26]:
import re 

# reshape quota file from wide to long
quota_long = coe_quota_df.melt(id_vars="DataSeries", var_name="month_label",value_name="value")
quota_long = quota_long.dropna(subset=["value"])
quota_long["value"] = pd.to_numeric(quota_long["value"], errors="coerce")
quota_long = quota_long.dropna(subset=["value"])

# parser to handle vehicle_class, measure, bidding_no
def parse_dataseries(s: str):
    s = s.strip()

    # Identify COE Category
    if "cars up to 1600cc" in s.lower():
        vehicle_class = "Category A"
        prefix = "Cars Up To 1600cc And 97kW"
    elif "cars above 1600cc" in s.lower():
        vehicle_class = "Category B"
        prefix = "Cars Above 1600cc Or 97kW"
    else:
        return pd.Series({"vehicle_class": None,
                          "measure": None,
                          "bidding_no": None})

    remaining = s.replace(prefix, "").strip(", ").strip()

    if "Quota Premium" in remaining:
        measure = "quota_premium"
    elif "Quota" in remaining:
        measure = "quota"
    elif "Successful Bids" in remaining:
        measure = "successful_bids"
    elif "Bids Received" in remaining:
        measure = "bids_received"
    else:
        measure = None

    bidding_no = None
    match = re.search(r"(\d)(st|nd) Bidding", remaining)
    if match:
        bidding_no = int(match.group(1))

    return pd.Series({
        "vehicle_class": vehicle_class,
        "measure": measure,
        "bidding_no": bidding_no
    })

parsed = quota_long["DataSeries"].apply(parse_dataseries)
quota_long = pd.concat([quota_long, parsed], axis=1)

quota_clean = quota_long[(quota_long['vehicle_class'].notna()) &(quota_long['measure'] == 'quota') &(quota_long['bidding_no'].notna())].copy()

quota_clean['value'] = pd.to_numeric(quota_clean['value'], errors='coerce')
quota_clean = quota_clean.dropna(subset=['value'])

quota_clean = quota_clean.rename(columns={"value": "quota"})
quota_clean['month_dt'] = pd.to_datetime(quota_clean['month_label'], format="%Y%b")
quota_clean = quota_clean[['month_dt','vehicle_class','bidding_no','quota']]

print("quota_clean...")
display(quota_clean.head())
print(quota_clean.shape)

quota_clean...


,month_dt,vehicle_class,bidding_no,quota
6,2025-10-01,Category A,1.0,1268.0
10,2025-10-01,Category A,2.0,1270.0
15,2025-10-01,Category B,1.0,830.0
19,2025-10-01,Category B,2.0,809.0
56,2025-09-01,Category A,1.0,1275.0


(1128, 4)


In [29]:
# merge quota + price_on (month_dt, vehicle_class, bidding_no)
coe_price_df["month_dt"] = pd.to_datetime(coe_price_df["month"])
merged = pd.merge(
    coe_price_df,
    quota_clean,
    on=["month_dt", "vehicle_class", "bidding_no"],
    how="inner"
)

print(merged.shape)
display(merged.sort_values(["month_dt","vehicle_class","bidding_no"]).head())

(748, 9)


,month,bidding_no,vehicle_class,quota_x,bids_success,bids_received,premium,month_dt,quota_y
0,2010-01,1,Category A,1152,1145,1342,18502,2010-01-01,1152.0
2,2010-01,2,Category A,1151,1149,1673,20501,2010-01-01,1151.0
1,2010-01,1,Category B,687,679,883,19190,2010-01-01,687.0
3,2010-01,2,Category B,717,717,1105,22400,2010-01-01,717.0
4,2010-02,1,Category A,1154,1153,1326,19989,2010-02-01,1154.0


### Model Building

#### Prediction - Build a model that forecasts COE prices (premium) for Category A and Category B for each bidding exercise

In [32]:
df = merged.copy()

df = df.drop(columns=["quota_x"])
df = df.rename(columns={"quota_y": "quota"})

# sort for lag calculations
df = df.sort_values(["vehicle_class", "month_dt", "bidding_no"]).reset_index(drop=True)

df["bids_received"] = pd.to_numeric(df["bids_received"], errors="coerce")
df["bids_success"] = pd.to_numeric(df["bids_success"], errors="coerce")
df["quota"] = pd.to_numeric(df["quota"], errors="coerce")
df["premium"] = pd.to_numeric(df["premium"], errors="coerce")
df = df.dropna(subset=["bids_received", "quota"])

# demand pressure = bids / quota
df["demand_pressure"] = df["bids_received"] / df["quota"]

# lag features (within each vehicle class)
df["premium_lag1"] = df.groupby("vehicle_class")["premium"].shift(1)
df["quota_lag1"]   = df.groupby("vehicle_class")["quota"].shift(1)

# time features
df["year"]  = df["month_dt"].dt.year
df["month"] = df["month_dt"].dt.month

# remove rows where lag features not available
df = df.dropna(subset=["premium_lag1", "quota_lag1", "demand_pressure"])
display(df.head())
print(df.shape)

,month,bidding_no,vehicle_class,bids_success,bids_received,premium,month_dt,quota,demand_pressure,premium_lag1,quota_lag1,year
1,1,2,Category A,1149.0,1673.0,20501,2010-01-01,1151.0,1.453519,18502.0,1152.0,2010
2,2,1,Category A,1153.0,1326.0,19989,2010-02-01,1154.0,1.149047,20501.0,1151.0,2010
3,2,2,Category A,1148.0,1493.0,20340,2010-02-01,1148.0,1.300523,19989.0,1154.0,2010
4,3,1,Category A,1141.0,1758.0,20802,2010-03-01,1148.0,1.531359,20340.0,1148.0,2010
5,3,2,Category A,1137.0,2183.0,28389,2010-03-01,1146.0,1.904887,20802.0,1148.0,2010


(667, 12)


In [33]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

models = {}

for cat in ["Category A", "Category B"]:
    print(f"\n===== Training model for {cat} =====")

    sub = df[df["vehicle_class"] == cat].copy()

    X = sub[["quota", "quota_lag1", "demand_pressure", "premium_lag1", "bidding_no", "year","month"]]

    y = sub["premium"]

    # Keep time order (no shuffle)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    model = GradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    models[cat] = {"model": model, "MAE": mae, "R²": r2}

    print(f"MAE: {mae:,.2f}")
    print(f"R² : {r2:.4f}")


===== Training model for Category A =====
MAE: 5,669.15
R² : 0.8450

===== Training model for Category B =====
MAE: 17,156.33
R² : -0.4696


#### Elasticity - Quantify how sensitive COE prices are to quota changes

In [36]:
!pip install statsmodels

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 1.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 2.5 MB/s eta 0:00:00a 0:00:01
DEPRECATION: pytorch-lightning 1.7.7 has a non-standard dependency specifier torch>=1.9.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [37]:
import statsmodels.api as sm
import numpy as np

elasticity_results = {}

for cat in ["Category A", "Category B"]:
    print(f"\n===== Elasticity estimation for {cat} =====")

    sub = df[df["vehicle_class"] == cat].copy()

    # Log transforms
    sub["log_price"] = np.log(sub["premium"])
    sub["log_quota"] = np.log(sub["quota"])
    sub["log_dpressure"] = np.log(sub["demand_pressure"])

    # Elasticity model
    X = sub[["log_quota", "log_dpressure"]]
    X = sm.add_constant(X)
    y = sub["log_price"]

    model = sm.OLS(y, X).fit()
    elasticity_results[cat] = model

    print(model.summary().tables[1])


===== Elasticity estimation for Category A =====
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            13.3293      0.227     58.778      0.000      12.883      13.775
log_quota        -0.3659      0.031    -11.694      0.000      -0.427      -0.304
log_dpressure    -0.0735      0.097     -0.760      0.448      -0.264       0.117

===== Elasticity estimation for Category B =====
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            14.4505      0.231     62.642      0.000      13.997      14.904
log_quota        -0.5114      0.034    -15.140      0.000      -0.578      -0.445
log_dpressure    -0.1961      0.104     -1.879      0.061      -0.401       0.009
